# Nuclei Lab: from image to measurement

This walkthrough uses real BBBC039 validation images. It is a classical computer vision pipeline, not a trained neural network. Benchmark results are produced by `scripts/benchmark.py` using separate test images.

In [ ]:
from pathlib import Path
import sys, json
import matplotlib.pyplot as plt
ROOT = Path.cwd() if (Path.cwd() / "cell_counter").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))
from cell_counter import Config, segment, measure, normalize, overlay
from cell_counter.dataset import read_image
image = read_image(ROOT / "examples/validation_01.tif")
print(image.shape, image.dtype)
plt.imshow(image, cmap="gray"); plt.axis("off");

## Separate touching nuclei

Otsu thresholding finds bright foreground. Watershed splits connected regions using distance-transform peaks. Load the frozen configuration selected on validation images.

In [ ]:
config = Config(**json.loads((ROOT / "reports/selected_config.json").read_text())["watershed"])
labels = segment(image, "watershed", config)
print("Detected nuclei:", labels.max())
plt.figure(figsize=(10, 6))
plt.imshow(overlay(image, labels)); plt.axis("off");

## Measure detected regions

Each row is one predicted nucleus. Pixel measurements are not physical measurements until microscope scale is known.

In [ ]:
table = measure(labels)
table.head()

## Read held-out results

This example is validation data. Generalization metrics come from the published 50-image test split. Pixel Dice alone does not measure correct instance counting.

In [ ]:
import pandas as pd
results = pd.read_csv(ROOT / "reports/summary.csv")
results[["method", "condition", "n_images", "absolute_count_error", "instance_f1", "foreground_dice"]]

## What to examine next

Open `reports/failure_analysis.png`: why do some nuclei merge or split? Read `docs/METHODS.md` before drawing conclusions about other microscopes or cell types.